In [ ]:
from google.colab import auth
from google.cloud import bigquery
import pandas as pd

auth.authenticate_user()

MY_PROJECT_ID = "YOUR-GCP-PROJECT-ID"
client = bigquery.Client(project=MY_PROJECT_ID)

query = """
SELECT
    ROW_ID,
    SUBJECT_ID,
    HADM_ID,
    CGID,
    TRIM(CATEGORY) AS CATEGORY,
    DESCRIPTION,
    TEXT
FROM `physionet-data.mimiciii_notes.noteevents`
WHERE TRIM(CATEGORY) = 'Physician'
  AND DESCRIPTION = 'Physician Resident Progress Note'
"""

df = client.query(query).to_dataframe()
print(f"Loaded {len(df)} physician notes.")
print(f"\nSub-types (DESCRIPTION breakdown):")
print(df['DESCRIPTION'].value_counts())
print(f"\nDistinct CGID authors: {df['CGID'].nunique()}")
print(f"\nNotes per author (top 20):")
print(df['CGID'].value_counts().head(20))

In [ ]:
# Resident Progress Notes only. Full cohort, no sampling.
subset = df[df['DESCRIPTION'] == 'Physician Resident Progress Note'].copy()
subset = subset.dropna(subset=['TEXT']).reset_index(drop=True)

print(f"Notes pulled              : {len(df)}")
print(f"After dropping empty TEXT : {len(subset)} (removed {len(df) - len(subset)})")
print(f"Distinct authors          : {subset['CGID'].nunique()}")


In [ ]:
"""
style_metrics.py
Physician Signature project — core feature functions.

Two INDEPENDENT axes are measured for every note:
  AXIS 1  Surface style   -> how the note is written (length, sentence shape)
  AXIS 2  Information load -> how much clinical content it actually carries

The whole thesis rests on these two axes NOT tracking each other.
Keeping them in separate functions makes that independence explicit and
defensible: we never let a surface feature leak into the content measure.
"""

import re

# --------------------------------------------------------------------------
# A transparent fixed dictionary is auditable: every match can be traced to a
# listed term, which is what defeats the circularity objection. The cost is
# coverage - no negation handling, no multi-word concepts, and a hard ceiling.
# This is a deliberate trade of sensitivity for auditability, not a claim of
# superiority over NER.
# --------------------------------------------------------------------------
CLINICAL_TERMS = {
    # symptoms
    "pain", "fever", "cough", "nausea", "vomiting", "dyspnea", "fatigue",
    "edema", "dizziness", "headache", "weakness", "bleeding", "rash",
    # findings / vitals
    "hypertension", "hypotension", "tachycardia", "bradycardia", "afebrile",
    "hypoxia", "murmur", "rales", "wheezing", "tenderness",
    # conditions
    "sepsis", "pneumonia", "failure", "infarction", "diabetes", "copd",
    "stroke", "embolism", "arrhythmia", "anemia", "infection",
    # meds / treatment
    "antibiotics", "insulin", "heparin", "warfarin", "furosemide",
    "morphine", "aspirin", "steroids", "diuretic", "anticoagulation",
    # actions
    "admitted", "discharged", "intubated", "transfused", "prescribed",
    "started", "discontinued", "titrated", "monitored", "consulted",
}


def tokenize(text):
    """Lowercase word tokens only. Shared by both axes so counts are comparable."""
    return re.findall(r"[a-zA-Z]+", str(text).lower())


# ---------------------------- AXIS 1: SURFACE STYLE ------------------------
def surface_features(text):
    """How the note is WRITTEN. Says nothing about clinical content."""
    tokens = tokenize(text)
    sentences = [s for s in re.split(r"[.!?]+", str(text)) if s.strip()]
    n_tokens = len(tokens)
    n_sent = max(len(sentences), 1)
    uniq = len(set(tokens))
    return {
        "word_count": n_tokens,
        "sentence_count": n_sent,
        "mean_sentence_len": round(n_tokens / n_sent, 2),
        "type_token_ratio": round(uniq / n_tokens, 4) if n_tokens else 0,  # lexical diversity
    }


# ------------------------- AXIS 2: INFORMATION LOAD ------------------------
def information_features(text):
    """
    Lexical content proxy: distinct terms from a fixed 50-term clinical
    dictionary appearing in the note. Measured only from the dictionary,
    never from length. This is a proxy, not a measure of clinical content:
    it has no negation handling, no multi-word terms, and saturates in long
    notes (observed ceiling 29 of 50 terms).
    'distinct_clinical_concepts' is the key number: how many DIFFERENT
    clinical things were documented, independent of how many words it took.
    """
    tokens = tokenize(text)
    present = [t for t in tokens if t in CLINICAL_TERMS]
    distinct = set(present)
    n_tokens = len(tokens)
    return {
        "clinical_term_hits": len(present),
        "distinct_clinical_concepts": len(distinct),
        # information DENSITY = signal per word. Low density = padding.
        "info_density": round(len(distinct) / n_tokens, 4) if n_tokens else 0,
    }


def all_features(text):
    """Convenience: both axes for one note, flattened into one dict."""
    f = {}
    f.update(surface_features(text))
    f.update(information_features(text))
    return f


In [ ]:
import numpy as np
import pandas as pd

# Feature functions are defined once, in the cell above. Not redefined here.
feats = subset['TEXT'].apply(all_features).apply(pd.Series)
f = pd.concat([subset[['ROW_ID', 'CGID']].reset_index(drop=True), feats], axis=1)

print("=" * 60)
print("HEADLINE EVIDENCE — Resident Progress Notes")
print("=" * 60)

wc = f['word_count']
print(f"Notes analysed:            {len(f)}")
print(f"Distinct authors:          {f['CGID'].nunique()}")
print(f"Word count  min/median/max:  {wc.min()} / {int(wc.median())} / {wc.max()}")
print(f"Longest note is {wc.max() / max(wc.min(),1):.1f}x the shortest.")

r_len_info = f['word_count'].corr(f['distinct_clinical_concepts'])
print(f"\nCorrelation (word_count vs distinct clinical concepts): r = {r_len_info:.3f}")
print(f"  -> r^2 = {r_len_info**2:.2f}: length explains ~{r_len_info**2*100:.0f}% of content variance; the rest is where author style lives.")

dens = f['info_density']
print(f"\nInfo density  min/median/max:  {dens.min():.3f} / {dens.median():.3f} / {dens.max():.3f}")
p10, p90 = dens.quantile(0.10), dens.quantile(0.90)
print(f"Info density 10th vs 90th percentile: {p10:.3f} vs {p90:.3f} (padded end vs dense end).")

# Fixed-content comparison, reported at multiple robustness levels.
# max/min is extreme-value driven (two notes decide it) and is reported only
# for comparison. p90/p10 is the headline.
target = int(f['distinct_clinical_concepts'].median())
same = f[f['distinct_clinical_concepts'] == target]

print("\n" + "=" * 60)
print(f"FIXED-CONTENT SPREAD (notes with exactly {target} distinct concepts)")
print("=" * 60)
print(f"n in this stratum: {len(same)}")

if len(same) >= 20:
    lo, hi = same['word_count'].min(), same['word_count'].max()
    p10 = same['word_count'].quantile(0.10)
    p25 = same['word_count'].quantile(0.25)
    p75 = same['word_count'].quantile(0.75)
    p90 = same['word_count'].quantile(0.90)

    print(f"  min / p10 / p25 / median / p75 / p90 / max:")
    print(f"  {lo:.0f} / {p10:.0f} / {p25:.0f} / {same['word_count'].median():.0f} "
          f"/ {p75:.0f} / {p90:.0f} / {hi:.0f}")
    print()
    print(f"  HEADLINE  p90/p10 ratio: {p90/max(p10,1):.2f}x")
    print(f"  robustness q75/q25 ratio: {p75/max(p25,1):.2f}x")
    print(f"  (max/min for comparison only: {hi/max(lo,1):.1f}x  <- extreme-value, do not headline)")
else:
    print(f"  Stratum too small (n={len(same)}) for stable percentiles.")

# Sensitivity: does the spread hold across other concept counts, not just the median?
print("\nSpread across concept strata (p90/p10):")
for c in sorted(f['distinct_clinical_concepts'].unique()):
    s = f[f['distinct_clinical_concepts'] == c]['word_count']
    if len(s) >= 50:
        a, b = s.quantile(0.10), s.quantile(0.90)
        print(f"  {c:>2} concepts (n={len(s):>5}): p10={a:>5.0f}  p90={b:>5.0f}  ratio={b/max(a,1):.2f}x")


In [ ]:
import matplotlib.pyplot as plt
import os

plt.style.use("seaborn-v0_8-whitegrid")
fig, ax = plt.subplots(2, 2, figsize=(13, 10))
fig.suptitle("Documentation Length vs Documented Clinical Content - Resident Progress Notes (n=62682)")

# Panel 1: scatter, length vs information
a = ax[0, 0]
a.scatter(f["word_count"], f["distinct_clinical_concepts"], alpha=0.5, edgecolor="k", linewidth=0.3)
m, b = np.polyfit(f["word_count"], f["distinct_clinical_concepts"], 1)
xs = np.linspace(f["word_count"].min(), f["word_count"].max(), 50)
a.plot(xs, m * xs + b, "r--", linewidth=2)
a.set_xlabel("Word count")
a.set_ylabel("Distinct clinical concepts")
a.set_title(f"Length vs. documented concepts  (r = {r_len_info:.2f}, r² = {r_len_info**2:.2f})")

# Panel 2: word count distribution
a = ax[0, 1]
a.hist(f["word_count"], bins=25, color="#3949ab", alpha=0.8, edgecolor="white")
a.axvline(f["word_count"].median(), color="red", linestyle="--", label=f"median = {int(f['word_count'].median())}")
a.set_xlabel("Word count"); a.set_ylabel("Number of notes")
a.set_title("Documentation length varies enormously")
a.legend()

# Panel 3: info density
a = ax[1, 0]
dens_cap = f["info_density"].quantile(0.995)
a.hist(f["info_density"].clip(upper=dens_cap), bins=40,
       color="#2e7d32", alpha=0.8, edgecolor="white")
a.set_xlim(0, dens_cap)
a.set_xlabel("Information density"); a.set_ylabel("Number of notes")
a.set_title("Signal-per-word: dense to heavily padded")

# Panel 4: same information, different length
a = ax[1, 1]
concepts = sorted(f["distinct_clinical_concepts"].unique())
data = [f[f["distinct_clinical_concepts"] == c]["word_count"].values for c in concepts]
bp = a.boxplot(data, labels=[str(int(c)) for c in concepts], showfliers=False)
# Thin the ticks so labels stop overlapping into unreadable strings
step = max(1, len(concepts) // 12)
for i, lbl in enumerate(a.get_xticklabels()):
    if i % step != 0:
        lbl.set_visible(False)
a.tick_params(axis='x', labelsize=9)
a.set_xlabel("Distinct clinical concepts (held constant)")
a.set_ylabel("Word count")
a.set_title("Same information, wildly different length")

plt.tight_layout(rect=[0, 0, 1, 0.97])
os.makedirs("figures", exist_ok=True)
plt.savefig("figures/physician_signature.png", dpi=140, bbox_inches="tight")
plt.show()
print("\nFigure saved to figures/physician_signature.png")

In [ ]:
# Can we even tie notes to authors, and is there repeat boilerplate per author?
rp = df[df['DESCRIPTION'] == 'Physician Resident Progress Note'].copy()

# How complete is authorship?
print(f"Notes total: {len(rp)}")
print(f"Notes WITH a CGID: {rp['CGID'].notna().sum()}")
print(f"Notes MISSING CGID: {rp['CGID'].isna().sum()}")
print(f"Distinct authors: {rp['CGID'].nunique()}")

# Pick the single most prolific author and look for repeated lines
top_author = rp['CGID'].value_counts().index[0]
author_notes = rp[rp['CGID'] == top_author]
print(f"\nMost prolific author {top_author}: {len(author_notes)} notes")

# Crude boilerplate check: split into lines, count lines that repeat across this author's notes
from collections import Counter
lines = []
for txt in author_notes['TEXT']:
    lines += [ln.strip() for ln in str(txt).split('\n') if len(ln.strip()) > 30]
common = Counter(lines).most_common(15)
print(f"\nTop repeated lines for this one author (boilerplate candidates):")
import hashlib
for line, count in common:
    h = hashlib.sha256(line.encode()).hexdigest()[:12]
    print(f"  [{count}x] len={len(line)} sha256={h}")

In [ ]:
# ============================================================
# STUDY 1 — PAPER NUMBERS. Copy this whole output block.
# ============================================================
wc = f['word_count']
dc = f['distinct_clinical_concepts']
dens = f['info_density']
r = wc.corr(dc)

print("=" * 60)
print("STUDY 1 NUMBERS FOR MANUSCRIPT")
print("=" * 60)
print(f"n_notes                  : {len(f)}")
print(f"n_authors                : {f['CGID'].nunique()}")
print(f"word_count min           : {wc.min()}")
print(f"word_count p25           : {wc.quantile(0.25):.0f}")
print(f"word_count median        : {wc.median():.0f}")
print(f"word_count p75           : {wc.quantile(0.75):.0f}")
print(f"word_count max           : {wc.max()}")
print(f"concepts min/median/max  : {dc.min()} / {dc.median():.0f} / {dc.max()}")
print(f"r (wc vs concepts)       : {r:.4f}")
print(f"r^2                      : {r**2:.4f}")
print(f"info_density p10         : {dens.quantile(0.10):.4f}")
print(f"info_density median      : {dens.median():.4f}")
print(f"info_density p90         : {dens.quantile(0.90):.4f}")
print(f"notes with ZERO concepts : {(dc == 0).sum()} ({(dc==0).mean()*100:.2f}%)")
print("=" * 60)

In [ ]:
import sys, platform
import pandas as pd, numpy as np, matplotlib
print("Python    :", sys.version.split()[0], "on", platform.system())
print("pandas    :", pd.__version__)
print("numpy     :", np.__version__)
print("matplotlib:", matplotlib.__version__)

In [ ]:
import pandas as pd

LOCKED_N, LOCKED_R = 62682, 0.6953
if len(f) != LOCKED_N:
    raise SystemExit(f"STOP: n={len(f)}, expected {LOCKED_N}. Re-run the query cell above first.")

ACTION_TERMS = {
    "admitted", "discharged", "intubated", "transfused", "prescribed",
    "started", "discontinued", "titrated", "monitored", "consulted",
}
CONTENT_ONLY = CLINICAL_TERMS - ACTION_TERMS
print(f"Full dictionary : {len(CLINICAL_TERMS)} terms")
print(f"Action verbs out: {len(ACTION_TERMS)} terms")
print(f"Content only    : {len(CONTENT_ONLY)} terms")

texts = subset["TEXT"].reset_index(drop=True)

def count_terms(text, vocab):
    return len(set(tokenize(text)) & vocab)

rebuilt_full = texts.apply(lambda t: count_terms(t, CLINICAL_TERMS))
if not (rebuilt_full.values == f["distinct_clinical_concepts"].reset_index(drop=True).values).all():
    raise SystemExit("STOP: rebuilt counts do not match the locked column. Run all from the top.")
print("Alignment check passed.")

concepts_no_actions = texts.apply(lambda t: count_terms(t, CONTENT_ONLY))
wc_b = f["word_count"].reset_index(drop=True)

r_full  = wc_b.corr(f["distinct_clinical_concepts"].reset_index(drop=True))
r_nouns = wc_b.corr(concepts_no_actions)

print("\n" + "=" * 64)
print("STUDY 1 SENSITIVITY: CONTENT PROXY WITHOUT ACTION VERBS")
print("=" * 64)
print(f"r, all 50 terms (locked)  : {r_full:.4f}   (expected {LOCKED_R})")
print(f"r, 40 clinical nouns only : {r_nouns:.4f}")
print(f"r-squared, nouns only     : {r_nouns**2:.4f}")
print(f"Change in r               : {r_nouns - r_full:+.4f}")
print(f"Median concepts, 50 terms : {f['distinct_clinical_concepts'].median():.0f}")
print(f"Median concepts, 40 nouns : {concepts_no_actions.median():.0f}")
print("=" * 64)

In [ ]:
print("len(subset):", len(subset))
print("len(f):     ", len(f))
print("CLINICAL_TERMS size:", len(CLINICAL_TERMS))

texts_chk = subset["TEXT"].reset_index(drop=True)
rebuilt_chk = texts_chk.apply(lambda t: len(set(tokenize(t)) & CLINICAL_TERMS))
locked_chk = f["distinct_clinical_concepts"].reset_index(drop=True)

diff = (rebuilt_chk != locked_chk)
print("mismatched rows:", diff.sum(), "of", len(diff))
if diff.sum() > 0:
    first_bad = diff.idxmax()
    print("first mismatch at row", first_bad)
    print("  rebuilt :", rebuilt_chk.iloc[first_bad])
    print("  locked  :", locked_chk.iloc[first_bad])
    print("  subset ROW_ID at that position:", subset["ROW_ID"].reset_index(drop=True).iloc[first_bad])
    print("  f ROW_ID at that position     :", f["ROW_ID"].reset_index(drop=True).iloc[first_bad] if "ROW_ID" in f.columns else "no ROW_ID column in f")

In [ ]:
import numpy as np, pandas as pd

LOCKED_N = 62682
if len(f) != LOCKED_N:
    raise SystemExit(f"STOP: n={len(f)}, expected {LOCKED_N}. Re-run the query cell above first.")

s1 = pd.DataFrame({
    "word_count": f["word_count"].reset_index(drop=True),
    "concepts":   f["distinct_clinical_concepts"].reset_index(drop=True),
})
s1["length_decile"] = pd.qcut(s1["word_count"], 10, labels=False, duplicates="drop") + 1

tab = s1.groupby("length_decile").agg(
    notes=("concepts", "size"),
    median_words=("word_count", "median"),
    median_concepts=("concepts", "median"),
    p90_concepts=("concepts", lambda x: x.quantile(0.90)),
)

print("=" * 64)
print("STUDY 1: CONTENT PROXY BY LENGTH DECILE (saturation check)")
print("=" * 64)
print(f"{'decile':>6} | {'notes':>6} | {'median words':>12} | {'median concepts':>15} | {'p90':>5}")
print("-" * 64)
for d, row in tab.iterrows():
    print(f"{int(d):>6} | {int(row['notes']):>6,} | {row['median_words']:>12,.0f} | "
          f"{row['median_concepts']:>15.0f} | {row['p90_concepts']:>5.0f}")
print("-" * 64)
print(f"Dictionary ceiling: {len(CLINICAL_TERMS)} terms. Observed maximum: {int(s1['concepts'].max())}.")
print(f"Notes at 20 or more concepts: {(s1['concepts'] >= 20).sum():,} "
      f"({(s1['concepts'] >= 20).mean() * 100:.2f}%)")
print("=" * 64)

In [ ]:
import numpy as np, pandas as pd, time

LOCKED_N, LOCKED_R = 62682, 0.6953
if len(f) != LOCKED_N:
    raise SystemExit(f"STOP: n={len(f)}, expected {LOCKED_N}. Re-run the query cell above first.")

work1 = pd.DataFrame({
    "CGID":       f["CGID"].reset_index(drop=True),
    "word_count": f["word_count"].reset_index(drop=True),
    "concepts":   f["distinct_clinical_concepts"].reset_index(drop=True),
})

r_point = work1["word_count"].corr(work1["concepts"])
print(f"Point estimate reproduced: r = {r_point:.4f}   (expected {LOCKED_R})")
if abs(r_point - LOCKED_R) > 0.0003:
    raise SystemExit(f"STOP: r = {r_point:.4f} does not match the locked {LOCKED_R}.")
print(f"Cohort guard passed: n = {len(work1):,}, authors = {work1['CGID'].nunique()}")

B, SEED = 1000, 2026
rng = np.random.default_rng(SEED)

groups = [(g["word_count"].to_numpy(dtype=float), g["concepts"].to_numpy(dtype=float))
          for _, g in work1.groupby("CGID", sort=False)]
n_auth = len(groups)
xs_all = [g[0] for g in groups]
ys_all = [g[1] for g in groups]

rs, t0 = [], time.time()
for b in range(B):
    idx = rng.integers(0, n_auth, size=n_auth)
    x = np.concatenate([xs_all[i] for i in idx])
    y = np.concatenate([ys_all[i] for i in idx])
    rs.append(np.corrcoef(x, y)[0, 1])

rs = np.asarray(rs)
lo, hi = np.percentile(rs, [2.5, 97.5])

print("\n" + "=" * 64)
print("STUDY 1: CLUSTER BOOTSTRAP 95% CI FOR r")
print("=" * 64)
print(f"Point estimate (locked, unchanged) : r = {r_point:.4f}")
print(f"95% CI, resampling physicians      : {lo:.4f} to {hi:.4f}")
print(f"Bootstrap mean / SD                : {rs.mean():.4f} / {rs.std(ddof=1):.4f}")
print(f"Replicates / seed / authors        : {B} / {SEED} / {n_auth}")
print(f"Runtime                            : {time.time() - t0:.0f} s")
print("=" * 64)
print(f"REPORT AS: r = {r_point:.3f} (95% CI {lo:.3f} to {hi:.3f})")

In [ ]:
# =====================================================================
# STUDY 1 VERIFICATION CELL
# Paste at the very bottom of the Study 1 notebook and run it LAST,
# after Restart and run all. It checks every locked Study 1 number at
# once and prints PASS or FAIL for each. Nothing is recomputed for the
# paper here; this only confirms the notebook is on the right cohort.
# =====================================================================
import numpy as np

checks, fails = [], 0

def check(label, got, want, tol=0):
    global fails
    ok = abs(float(got) - float(want)) <= tol
    if not ok:
        fails += 1
    checks.append((label, got, want, "PASS" if ok else "FAIL"))

wc = f["word_count"]
dc = f["distinct_clinical_concepts"]
r  = wc.corr(dc)

check("notes",                    len(f),                     62682)
check("authors (CGID)",           f["CGID"].nunique(),        435)
check("word count min",           wc.min(),                   1)
check("word count p25",           round(wc.quantile(0.25)),   544)
check("word count median",        round(wc.median()),         720)
check("word count p75",           round(wc.quantile(0.75)),   897)
check("word count max",           wc.max(),                   2745)
check("concepts min",             dc.min(),                   0)
check("concepts median",          round(dc.median()),         11)
check("concepts max",             dc.max(),                   29)
check("r",                        r,                          0.6953, 0.0003)
check("r squared",                r ** 2,                     0.4835, 0.0005)
check("zero-concept notes",       int((dc == 0).sum()),       36)
check("notes under 10 words",     int((wc < 10).sum()),       18)

# the fixed-content stratum the paper headlines
same = f[dc == 11]
check("stratum n at 11 concepts", len(same),                  5999)
p10, p90 = same["word_count"].quantile(0.10), same["word_count"].quantile(0.90)
p25, p75 = same["word_count"].quantile(0.25), same["word_count"].quantile(0.75)
check("p90/p10 at 11 concepts",   p90 / max(p10, 1),          1.93, 0.005)
check("q75/q25 at 11 concepts",   p75 / max(p25, 1),          1.41, 0.005)

# the boilerplate author
top_author = f["CGID"].value_counts().idxmax()
check("most prolific author id",  top_author,                 19458)
check("notes by that author",     f["CGID"].value_counts().max(), 968)

print("=" * 66)
print("STUDY 1 LOCKED-NUMBER VERIFICATION")
print("=" * 66)
print(f"{'quantity':28s} | {'got':>12s} | {'expected':>10s} | result")
print("-" * 66)
for label, got, want, res in checks:
    got_s = f"{got:,.4f}" if isinstance(got, float) else f"{got:,}"
    print(f"{label:28s} | {got_s:>12s} | {want:>10,} | {res}")
print("-" * 66)
if fails == 0:
    print("ALL CHECKS PASSED. The notebook is on the locked cohort.")
else:
    print(f"{fails} CHECK(S) FAILED.")
    print("If 'notes' is 62,569 and 'authors' is 434, the query still has")
    print("AND ISERROR IS NULL in it. Delete that line and run all again.")
print("=" * 66)
print("PASTE THIS WHOLE TABLE BACK.")